# 04b — Tuning Hyperparameter RM-b (Frozen Encoder + Head) — versi **colab**
Eksplorasi adaptif; head dilatih di atas fitur beku ter-cache (tiap trial hitungan detik).

### Metodologi: eksplorasi adaptif (coordinate line-search)

Bukan grid buta. Mulai dari **baseline**, lalu tune satu hyperparameter menurut **urutan dampak**; tiap langkah **probe arah** (naik/turun) dan lanjut selama **val F1-macro** membaik (early-stop `eps=0.001`). HP berikutnya memakai config yang sudah diperbarui. **Setiap konfigurasi tersimpan** sebagai satu trial + `rationale` (alasan). Seleksi di validation; **test dievaluasi sekali** untuk config final. Semua hasil → `results/tuning/`.

## 1. Setup

In [ ]:
import os, sys
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB=True
except Exception:
    IN_COLAB=False
if IN_COLAB:
    os.system('pip install -q -U transformers scikit-learn')
os.environ.setdefault('MPLBACKEND','Agg')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/IndoBERT-with-RAC') if IN_COLAB else Path('..')
sys.path.insert(0, str((PROJECT_DIR/'src').resolve()))
import numpy as np, pandas as pd, torch
import tuning as T
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| Colab' if IN_COLAB else '| lokal')

## 2. Ruang pencarian & baseline

In [ ]:
OUT_DIR = PROJECT_DIR / 'results' / 'tuning'
FEATURES_DIR = PROJECT_DIR / 'results/features'
DATA_DIR = PROJECT_DIR / 'dataset' / 'splits'
META = PROJECT_DIR / 'dataset' / 'processed' / 'metadata.json'
ORDER  = ['epochs','lr','dropout','head_arch','weight_decay']   # epochs dulu (fix underfit)
SPACES = {'epochs':[5,10,20,40,80], 'lr':[5e-5,1e-4,2e-4,5e-4,1e-3],
          'dropout':[0.0,0.1,0.3,0.5], 'head_arch':['linear','mlp'], 'weight_decay':[0.0,0.01,0.1]}
BASELINE = {'epochs':5,'lr':2e-4,'dropout':0.1,'head_arch':'linear','weight_decay':0.01,'batch':32,'seed':42}
print('order', ORDER); print('baseline', BASELINE)

## 3. Muat fitur beku (ekstrak bila belum ada)

In [ ]:
import json
def load_feats():
    return {s:(np.load(FEATURES_DIR/f'{s}_emb.npy'), np.load(FEATURES_DIR/f'{s}_label.npy')) for s in ['train','val','test']}
if not all((FEATURES_DIR/f'{s}_emb.npy').exists() for s in ['train','val','test']):
    print('Fitur belum ada -> ekstrak dengan encoder beku...')
    import modeling as M
    from dataset import load_tokenizer, GamblingCommentDataset
    from torch.utils.data import DataLoader
    FEATURES_DIR.mkdir(parents=True, exist_ok=True)
    tok = load_tokenizer('indobenchmark/indobert-base-p2'); enc = M.build_encoder(tok).to(device)
    for s in ['train','val','test']:
        df = pd.read_csv(DATA_DIR/f'{s}.csv')
        dl = DataLoader(GamblingCommentDataset(df['text_clean'], df['label'], tokenizer=tok, max_length=128), batch_size=32)
        e,l = M.extract_features(enc, dl, device); np.save(FEATURES_DIR/f'{s}_emb.npy', e); np.save(FEATURES_DIR/f'{s}_label.npy', l)
feats = load_feats()
cw = json.load(open(META, encoding='utf-8'))['class_weights']
weight = torch.tensor([cw['0'], cw['1']], dtype=torch.float, device=device)
print('fitur:', {s:feats[s][0].shape for s in feats})

## 4. Jalankan tuning adaptif (murah)

In [ ]:
final_cfg, decisions, test_metrics = T.tune_rmb(feats, weight, device, out_dir=str(OUT_DIR),
                                                spaces=SPACES, order=ORDER, baseline=BASELINE, eps=0.001)

## 5. Hasil, jejak keputusan, metrik test

In [ ]:
trials = pd.read_csv(OUT_DIR / 'rmb_trials.csv')
cols = [c for c in ['trial_id','stage','val_f1_macro','val_f1_judi','best_epoch','train_time_s','rationale'] if c in trials.columns]
print(f'Total trial: {len(trials)}')
print(trials[cols].to_string(index=False))
print('\n=== Jejak keputusan ===')
for d in decisions:
    print(f"  [{d['stage']}] -> {d['chosen']}  (val F1 {d['val_f1_macro']:.4f})  | {d['note']}")
print('\n=== Config final ===', final_cfg)
print('=== Metrik TEST final ===')
for k in ['accuracy','f1_macro','precision_macro','recall_macro','f1_class1','precision_class1','recall_class1']:
    print(f'  {k:18s}: {test_metrics[k]:.4f}')
print('\nArtefak tersimpan di', OUT_DIR)

## Ringkasan
Semua trial + kurva + checkpoint head terbaik + `tuning_summary.json` di `results/tuning/`. Karena RM-b murah, pencarian dilakukan lebih dalam (epochs/lr/dropout/arsitektur head).